# Data preparation, 2018 to 2026

This notebook cleans every useful file in the folder. The project window is 2018 through 2026. Where NISR sent the same interviews as both Stata and SPSS, only one copy is read. The duplicate SPSS archives are not part of this notebook.

| Year | What is prepared |
| --- | --- |
| 2018 | Seasonal survey, and the food-security survey |
| 2019 | Seasonal survey |
| 2020 | Seasonal survey |
| 2021 | Seasonal survey |
| 2022 | Seasonal survey |
| 2023 | Seasonal survey |
| 2024 | Seasonal survey, and the household survey |
| 2025 | Seasonal survey |
| 2026 | Published seasonal tables for seasons A and B |

Prices, monthly rainfall, and the national export file are prepared beside the surveys. They are not tied to a single season.

Open this notebook in Google Colab and choose **Runtime → Run all**. The files need to sit in `My Drive/Rwanda intelligence analysis`.

A few words that come up all the way through:

- A **segment** is the piece of land the seasonal survey sampled. A **plot** is one field inside it. A plot can grow more than one crop, so the crop file has more rows than the fertilizer file.
- A **household** is one home. That is the unit in the 2018 food-security survey and in the 2024 household survey.
- The **weight** says how many similar plots, or households, a row stands for. A national total uses the weight. A plain sum of the rows does not.

`STATA.zip` is the 2019 seasonal survey. Season A runs from September 2018 to February 2019, season B from March to June 2019, and season C in July and August 2019. The Stata file was saved on 2 October 2020. That date is the export date, not the survey year.

Sentinel-2 vegetation is not in the folder, so it is not prepared. The trade file is the national export total for each year, not exports of each crop.


In [ ]:
# Find the project folder and the tool that reads Stata files.
# In Colab the folder is on Google Drive. Outside Colab it is this project on the computer.

from pathlib import Path
import csv
import io
import re
import subprocess
import sys
import zipfile
from collections import defaultdict

import numpy as np
import pandas as pd
from IPython.display import display

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    drive = None
    files = None
    IN_COLAB = False

try:
    import pyreadstat
except ImportError:
    # Stata files need this package. Colab does not include it, so we install it once.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyreadstat", "openpyxl"])
    import pyreadstat

if IN_COLAB:
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/Rwanda intelligence analysis")
else:
    ROOT = Path.cwd()
    if not (ROOT / "data" / "source" / "Microdata_SAS2023.zip").exists() and not (ROOT / "Microdata_SAS2023.zip").exists():
        ROOT = Path(r"c:\Users\Musanze\Desktop\Evidances\Rwanda intelligence analysis")

RAW = ROOT / "data" / "raw"
OUT = ROOT / "data" / "processed"
SOURCE = ROOT / "data" / "source"
RAW.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
SOURCE.mkdir(parents=True, exist_ok=True)

def locate(name):
    # The original downloads sit together in data/source.
    chosen = SOURCE / name
    if chosen.exists():
        return chosen
    return ROOT / name

pd.set_option("display.max_columns", 16)
pd.set_option("display.width", 120)
pd.set_option("display.max_rows", 30)

print("Project folder:", ROOT)


## Unpack the files

Each survey is stored once. Stata is used when the answers are still stored as codes, because the file carries the wording. CSV is used for 2023, 2024, and 2025, because those answers are already written as words. SPSS copies are left unread. A file with nothing in it is skipped.


In [ ]:
# These are the archives the project needs. Duplicate SPSS archives are not on the list.
REQUIRED = [
    "rwa-nisr-sas-2018-stata (1).zip",
    "STATA.zip",
    "Microdata for SAS 2020.zip",
    "Microdata_SAS2021.zip",
    "Microdata_SAS_2022.zip",
    "Microdata_SAS2023.zip",
    "Dataset (2).zip",
    "Microdata (7).zip",
    "Tables_2026 Season A (1).xlsx",
    "Tables_2026 Season B (1).xlsx",
    "rwa_adm_2006_nisr_wgs1984_20181002_shp.zip",
    "train-00000-of-00001.parquet",
    "test-00000-of-00001.parquet",
    "hingasmart_rwanda_chirps_monthly_district_rainfall_2018_2026.csv",
]

missing = [name for name in REQUIRED if not locate(name).exists()]
if IN_COLAB and missing:
    print("These files are not in the Drive folder yet. Choose them below.")
    for name in missing:
        print(" -", name)
    for name, payload in files.upload().items():
        raw = payload if isinstance(payload, (bytes, bytearray)) else bytes(payload)
        (SOURCE / Path(name).name).write_bytes(raw)
        print("Saved", Path(name).name)

still_missing = [name for name in REQUIRED if not locate(name).exists()]
if still_missing:
    raise FileNotFoundError("Still missing: " + ", ".join(still_missing))


def landed_name(filename):
    # Keep the season folder in the name. One 2024 season C file was saved with a season B name.
    parts = filename.replace("\\", "/").split("/")
    parent = parts[-2] if len(parts) > 1 else ""
    if "season" in parent.lower():
        return parent.replace(" ", "_") + "__" + parts[-1]
    return parts[-1]


def unpack(zip_path, dest, keep):
    # Skip a file that is already unpacked at the same size, so a second run is quick.
    dest.mkdir(parents=True, exist_ok=True)
    fresh = 0
    skipped = 0
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            filename = info.filename.replace("\\", "/")
            if info.is_dir() or filename.startswith("__MACOSX") or "/._" in filename:
                continue
            if not keep(filename):
                continue
            if info.file_size < 1000:
                skipped += 1
                continue
            target = dest / landed_name(filename)
            if target.exists() and target.stat().st_size == info.file_size:
                continue
            target.write_bytes(zf.read(info))
            fresh += 1
    return fresh, skipped


def stata_only(filename):
    low = filename.lower()
    return low.endswith(".dta") and "/spss/" not in low


def csv_only(filename):
    return filename.lower().endswith(".csv")


jobs = [
    ("rwa-nisr-sas-2018-stata (1).zip", RAW / "sas" / "2018", stata_only),
    ("STATA.zip", RAW / "sas" / "2019", stata_only),
    ("Microdata for SAS 2020.zip", RAW / "sas" / "2020", stata_only),
    ("Microdata_SAS2021.zip", RAW / "sas" / "2021", stata_only),
    ("Microdata_SAS_2022.zip", RAW / "sas" / "2022", stata_only),
    ("Microdata_SAS2023.zip", RAW / "sas" / "2023", csv_only),
    ("Dataset (2).zip", RAW / "sas" / "2024", csv_only),
    ("Microdata (7).zip", RAW / "sas" / "2025", csv_only),
    ("rwa_adm_2006_nisr_wgs1984_20181002_shp.zip", RAW / "boundaries", lambda name: Path(name).suffix.lower() in {".shp", ".dbf", ".shx", ".prj", ".cpg"}),
]

for zip_name, dest, keep in jobs:
    added, skipped = unpack(locate(zip_name), dest, keep)
    count = len([item for item in dest.iterdir() if item.is_file()])
    note = f"{zip_name}: {count} files ready"
    if added:
        note += f", {added} just unpacked"
    if skipped:
        note += f", {skipped} empty files skipped"
    print(note)

## District names

People say "South" or "Southern Province", and a district is sometimes saved with a code next to the name. The boundary file has one official name and one code for each of the 30 districts. We keep the original answer, and we add the official name whenever we can recognise it.


In [ ]:
# Read the district table that sits beside the map. We only need the names and codes here.
def read_dbf(path):
    data = Path(path).read_bytes()
    n_records = int.from_bytes(data[4:8], "little")
    header_len = int.from_bytes(data[8:10], "little")
    record_len = int.from_bytes(data[10:12], "little")
    fields, offset = [], 32
    while data[offset] != 0x0D:
        fields.append((data[offset:offset + 11].split(b"\x00", 1)[0].decode("ascii"), data[offset + 16]))
        offset += 32
    rows = []
    for i in range(n_records):
        record = data[header_len + i * record_len: header_len + (i + 1) * record_len]
        if not record or record[0] == 0x2A:
            continue
        pos, row = 1, {}
        for name, length in fields:
            row[name] = record[pos:pos + length].decode("latin1").strip()
            pos += length
        rows.append(row)
    return pd.DataFrame(rows)

adm2_path = next(iter(list((RAW / "boundaries").glob("*adm2*.dbf")) or list((RAW / "boundaries").glob("*ADM2*.dbf"))))
adm2 = read_dbf(adm2_path)
districts = pd.DataFrame({
    "district_key": adm2["ADM2_EN"].str.lower().str.strip(),
    "district": adm2["ADM2_EN"].str.strip(),
    "district_pcode": adm2["ADM2_PCODE"].str.strip(),
    "province": adm2["ADM1_EN"].str.strip(),
    "province_code": adm2["ADM1_PCODE"].str.strip(),
}).drop_duplicates("district_key")

DISTRICT_LOOKUP = {row.district_key: row for row in districts.itertuples(index=False)}
DISTRICTS_LONGEST_FIRST = sorted(DISTRICT_LOOKUP, key=len, reverse=True)
print(f"{len(districts)} districts.")
display(districts.sort_values(["province_code", "district"]).reset_index(drop=True))


## How the cleaning works

The same rules are used on every file.

Blank cells, `NA`, a lone full stop, and the mistyped phrase `Missin value` become real missing values. In the Stata files, numbered answers are replaced by their labels, so you see "Male" rather than 1. Measurements such as area, yield and the weight stay as numbers.

The original question codes stay in the file. Plain columns are added in front: year, place, weight, and, where the question really is a crop, the crop name. In 2018 and 2019 the crop name is question `s2q6`. From 2020 it is `s2q4`.

The 2018 harvested-area question is in square metres. `harvested_area_ha` divides that one column by 10,000 so it can sit beside the later years, which are already in hectares. The original column is kept. Plot size in `s2q2` is not converted, because its unit is not the same in every file.

Rows are not deleted. A district we cannot recognise is flagged in `province_check` and kept. A column that is empty in every row of a table is dropped, because it adds nothing to that table.

`plot_weight` belongs to the seasonal survey. `sample_weight` belongs to the household and food-security surveys. They are not interchangeable. Where both harvested hectares and yield are present, `production_kg` is their product. Yield in these files is kilograms per hectare. Where yield is missing and the questionnaire already recorded the harvest in kilograms, that recorded quantity is used instead.


In [ ]:
# Shared tools. Each one does one job, and the surveys below all call the same ones.

MISSING_TEXT = {"", "na", "n/a", "n.a.", ".", "missin value", "missing value", "missing", "nan", "none"}
WEIGHT_COLUMNS = ["plot_weight", "Plot_weight", "weight_plot", "weight_plot_final", "finalplot_weight", "WH_Plot", "Weight", "weight"]
HARVEST_HA_COLUMNS = ["Harv_area", "Harvested_Area", "Harvested_Crop_Area_ha"]
STRATUM_LABEL = {
    "intensive cropland on hillside": "Intensive cropland on hillsides",
    "large scale farmer": "Large scale farm",
    "large scale farmer/lsf": "Large scale farm",
    "lsf": "Large scale farm",
    "ssf": "Small scale farm",
}
KNOWN_LABELS = {
    "s1q1": "Province",
    "s1q2": "District",
    "s1q3": "Stratum",
    "s1q8": "Sex of the farmer",
    "s1q9": "Age of the farmer",
    "s2q4": "Crop name from 2020 onward. In 2018 and 2019 this question is the cropping system, and the crop name is s2q6.",
    "s2q6": "Crop name in 2018 and 2019. In later years this question is no longer the crop name.",
    "s3q1": "Crop name on the crop-screening files.",
    "yield": "Yield, kilograms per hectare",
    "hhid": "Household id",
    "PARENT_KEY": "Household id in the 2018 food-security files",
    "FinalWeight": "Household weight",
    "S0_C_Prov": "Province",
    "S0_D_Dist": "District",
}
FRONT_COLUMNS = [
    "survey", "year", "season", "module", "source_file", "segment_id", "farm_scale",
    "province", "province_code", "province_survey", "district", "district_pcode", "district_survey",
    "province_check", "stratum", "plot_id", "plot_weight", "farmer_type", "sex", "age",
    "crop_name", "crop_category", "harvested_area_ha", "crop_area_ha", "yield_kg_per_ha", "production_kg",
]


def blank_to_missing(value):
    # Turn the various ways a file says "no answer" into one missing value.
    if value is None:
        return pd.NA
    try:
        if pd.isna(value):
            return pd.NA
    except (TypeError, ValueError):
        pass
    text = re.sub(r"\s+", " ", str(value)).strip()
    if text.lower() in MISSING_TEXT:
        return pd.NA
    return text


def clean_text_columns(frame):
    out = frame.copy()
    for column in out.select_dtypes(include=["object", "string"]).columns:
        out[column] = out[column].map(blank_to_missing).astype("string")
    return out


def as_id(series):
    # Keep identifiers as text so 12001 does not become 12001.0 and then fail to match.
    cleaned = []
    for value in series.tolist():
        try:
            missing = value is None or bool(pd.isna(value))
        except (TypeError, ValueError):
            missing = False
        if missing:
            cleaned.append(pd.NA)
            continue
        if isinstance(value, float) and value.is_integer():
            cleaned.append(str(int(value)))
        elif isinstance(value, int):
            cleaned.append(str(value))
        else:
            text = str(value).strip()
            if text.endswith(".0") and text[:-2].lstrip("-").isdigit():
                text = text[:-2]
            cleaned.append(pd.NA if not text or text.lower() in MISSING_TEXT else text)
    return pd.Series(cleaned, index=series.index, dtype="string")


def first_present(frame, names):
    # Use the first of these columns that actually has an answer.
    result = pd.Series(pd.NA, index=frame.index, dtype="object")
    for name in names:
        if name in frame.columns:
            result = result.where(result.notna(), frame[name])
    return result


def resolve_province(value):
    try:
        if value is pd.NA or pd.isna(value):
            return "", ""
    except (TypeError, ValueError):
        return "", ""
    text = str(value).strip().lower()
    if not text or text in MISSING_TEXT:
        return "", ""
    if "kigali" in text or "umujyi" in text:
        return "Kigali City", "RW1"
    if "eastern" in text or "iburasirazuba" in text or text in {"east", "est"}:
        return "Eastern Province", "RW5"
    if "western" in text or "iburengerazuba" in text or text in {"west", "ouest"}:
        return "Western Province", "RW3"
    if "northern" in text or "amajyaruguru" in text or text in {"north", "nord"}:
        return "Northern Province", "RW4"
    if "southern" in text or "amajyepfo" in text or text in {"south", "sud"}:
        return "Southern Province", "RW2"
    return "", ""


def resolve_district(value):
    # Accept "Musanze", "Musanze - 27", or a name with the code in front of it.
    try:
        if value is pd.NA or pd.isna(value):
            return None
    except (TypeError, ValueError):
        return None
    text = re.sub(r"\s+", " ", str(value)).strip().lower()
    if not text or text in MISSING_TEXT:
        return None
    text = text.replace("nyarugenege", "nyarugenge")
    if text in DISTRICT_LOOKUP:
        return DISTRICT_LOOKUP[text]
    for key in DISTRICTS_LONGEST_FIRST:
        if key in text:
            return DISTRICT_LOOKUP[key]
    return None


def attach_geography(frame, province_column, district_column):
    # Keep the recorded place, and add the official district when the name is recognisable.
    province_raw = frame[province_column] if province_column in frame.columns else pd.Series(pd.NA, index=frame.index)
    district_raw = frame[district_column] if district_column in frame.columns else pd.Series(pd.NA, index=frame.index)
    frame["province_survey"] = province_raw.map(blank_to_missing).astype("string")
    frame["district_survey"] = district_raw.map(blank_to_missing).astype("string")
    # Lists, not Series.map: a district match is a small record, and map would turn that record into an array.
    official = [resolve_district(value) for value in frame["district_survey"].tolist()]
    survey_province = [resolve_province(value) for value in frame["province_survey"].tolist()]

    frame["district"] = pd.Series(["" if row is None else row.district for row in official], index=frame.index, dtype="string")
    frame["district_pcode"] = pd.Series(["" if row is None else row.district_pcode for row in official], index=frame.index, dtype="string")
    frame["province"] = pd.Series(
        [(row.province if row is not None else pair[0]) for row, pair in zip(official, survey_province)],
        index=frame.index, dtype="string",
    )
    frame["province_code"] = pd.Series(
        [(row.province_code if row is not None else pair[1]) for row, pair in zip(official, survey_province)],
        index=frame.index, dtype="string",
    )

    checks = []
    recorded_values = frame["district_survey"].tolist()
    survey_values = frame["province_survey"].tolist()
    for recorded, survey_name, row in zip(recorded_values, survey_values, official):
        recorded_blank = pd.isna(recorded) or str(recorded) == ""
        survey_blank = pd.isna(survey_name) or str(survey_name) == ""
        if recorded_blank:
            checks.append("blank_district")
        elif row is None:
            checks.append("district_not_in_boundary")
        elif survey_blank:
            checks.append("blank_survey_province")
        elif resolve_province(survey_name)[0] not in ("", row.province):
            checks.append("survey_province_disagrees")
        else:
            checks.append("ok")
    frame["province_check"] = checks
    return frame


def harmonize_stratum(series):
    text = series.map(lambda value: "" if pd.isna(value) else str(value).strip()).astype("string")
    extracted = text.str.extract(r"^(\d+)\s+(.+)$")
    label = extracted[1].fillna(text).str.strip()
    return label.str.lower().map(STRATUM_LABEL).fillna(label).astype("string")


def read_stata(path):
    # apply_value_formats writes "Male" instead of leaving the code 1.
    # A few household files mix in a Windows character, so those are read a second time in that encoding.
    try:
        frame, meta = pyreadstat.read_dta(str(path), apply_value_formats=True)
    except UnicodeDecodeError:
        frame, meta = pyreadstat.read_dta(str(path), apply_value_formats=True, encoding="cp1252")
    labels = list(meta.column_labels or [])
    if len(labels) < frame.shape[1]:
        labels += [""] * (frame.shape[1] - len(labels))
    for column in frame.select_dtypes(include="category").columns:
        frame[column] = frame[column].astype("object")
    return clean_text_columns(frame), labels


def season_of(filename):
    compact = re.sub(r"[^a-z0-9]", "", filename.lower())
    for letter, token in (("A", "seasona"), ("B", "seasonb"), ("C", "seasonc")):
        if token in compact:
            return letter
    return ""


def module_of(filename):
    low = filename.lower()
    if "crop production" in low or "production" in low:
        return "crop_production"
    if "pesticide" in low and "fertilizer" not in low:
        return "pesticide"
    if "fertilizer" in low:
        return "fertilizer"
    if "land tenure" in low or "land_tenure" in low:
        return "land_tenure"
    if "agricultural practice" in low or "agricultural_practice" in low or "irrigation" in low or "soil_preparation" in low:
        return "agricultural_practice"
    if "agroforestry" in low:
        return "agroforestry"
    if "antierosion" in low or "erosion" in low:
        return "erosion"
    if "screening_crops" in low or "screening-crops" in low:
        return "screening_crops"
    if "screening" in low:
        return "screening"
    return "other"


def farm_scale_of(filename):
    low = filename.lower()
    if "_lsf_" in low or "_lsf." in low:
        return "Large scale farm"
    if "_ssf_" in low or "_ssf." in low:
        return "Small scale farm"
    return ""


def order_columns(frame, front):
    leading = [column for column in front if column in frame.columns]
    return frame[leading + [column for column in frame.columns if column not in leading]]


def prepare_seasonal_file(frame, year, season, module, filename):
    # Add the plain columns. The questionnaire columns stay where they are.
    out = frame.copy()
    out.insert(0, "source_file", filename)
    out.insert(0, "module", module)
    out.insert(0, "season", season)
    out.insert(0, "year", year)
    out.insert(0, "survey", "SAS")

    id_column = next((name for name in ("Segment_ID", "segment_id", "s1q0") if name in out.columns), None)
    out.insert(5, "segment_id", as_id(out[id_column]) if id_column else pd.Series(pd.NA, index=out.index, dtype="string"))
    out = attach_geography(out, "s1q1", "s1q2")
    out["stratum"] = harmonize_stratum(out["s1q3"] if "s1q3" in out.columns else pd.Series(pd.NA, index=out.index))
    out["plot_id"] = as_id(out["s2q1"]) if "s2q1" in out.columns else pd.Series(pd.NA, index=out.index, dtype="string")
    out["plot_weight"] = pd.to_numeric(first_present(out, WEIGHT_COLUMNS), errors="coerce")

    # On the screening files, s1q7 is the number of grids, not the type of farmer.
    farmer_sources = ["s1q7", "s2q3_6"] if module in {"crop_production", "fertilizer", "agricultural_practice", "land_tenure"} else ["s2q3_6"]
    out["farmer_type"] = first_present(out, farmer_sources).map(blank_to_missing).astype("string")
    out["sex"] = out["s1q8"].map(blank_to_missing).astype("string") if "s1q8" in out.columns else pd.Series(pd.NA, index=out.index, dtype="string")
    out["age"] = pd.to_numeric(out["s1q9"], errors="coerce") if "s1q9" in out.columns else np.nan

    out["farm_scale"] = farm_scale_of(filename)

    # In 2018 and 2019, s2q4 is the cropping system and s2q6 is the crop. From 2020, s2q4 is the crop.
    if module == "crop_production" and year <= 2019 and "s2q6" in out.columns:
        out["crop_name"] = out["s2q6"].map(blank_to_missing).astype("string")
    elif module == "crop_production" and "s2q4" in out.columns:
        out["crop_name"] = out["s2q4"].map(blank_to_missing).astype("string")
    elif module == "screening_crops" and "s3q1" in out.columns:
        out["crop_name"] = out["s3q1"].map(blank_to_missing).astype("string")
    else:
        out["crop_name"] = pd.Series(pd.NA, index=out.index, dtype="string")
    out["crop_category"] = first_present(out, ["CropCategory", "CropGroup", "crop_group"]).map(blank_to_missing).astype("string")

    harvested = pd.to_numeric(first_present(out, HARVEST_HA_COLUMNS), errors="coerce")
    if module == "crop_production" and harvested.isna().all() and "Harvested_area" in out.columns:
        # The 2018 question is in square metres. Later years are already in hectares.
        harvested = pd.to_numeric(out["Harvested_area"], errors="coerce") / 10000
    if module == "crop_production" and harvested.isna().all():
        harvested = pd.to_numeric(first_present(out, ["Crop_Area", "Crop_Area_ha"]), errors="coerce")
    if module != "crop_production":
        harvested = pd.Series(np.nan, index=out.index)
    out["harvested_area_ha"] = harvested
    if module in {"screening_crops", "screening"}:
        out["crop_area_ha"] = pd.to_numeric(first_present(out, ["Crop_Area", "crop_Area", "crop_area", "Crop_Area_ha"]), errors="coerce")
    else:
        out["crop_area_ha"] = np.nan
    yield_name = next((column for column in out.columns if str(column).lower() == "yield"), None)
    yiel = pd.to_numeric(out[yield_name], errors="coerce") if yield_name else pd.Series(np.nan, index=out.index)
    out["yield_kg_per_ha"] = yiel
    production = harvested * yiel
    if module == "crop_production" and "s2q21" in out.columns:
        reported_kg = pd.to_numeric(out["s2q21"], errors="coerce")
        production = production.where(production.notna(), reported_kg)
    out["production_kg"] = production.round(6)
    return order_columns(out, FRONT_COLUMNS)


def dictionary_rows(survey, year, season, module, filename, columns, labels):
    rows = []
    for column, label in zip(columns, labels):
        rows.append({
            "survey": survey,
            "year": year,
            "season": season,
            "module": module,
            "file": filename,
            "column": column,
            "question": (label or "").strip() or KNOWN_LABELS.get(column, ""),
        })
    return rows


print("Cleaning tools are ready.")


## Seasonal survey, 2018 to 2025

Each year has three seasons. Season A is September to February, season B is March to June, and season C is the short season in marshland, so it is a much smaller file.

The pieces of the questionnaire are stacked across years: crop production, crop screening, fertilizer, pesticide (2018 keeps pesticides in their own file), farming practice, agroforestry, erosion, and land tenure where that section was asked. In 2018 the small-scale and large-scale farms are separate files. `farm_scale` records which file a row came from.

`crop_name` comes from `s2q6` in 2018 and 2019, and from `s2q4` after that. On the 2023 to 2025 files those values are real crop names, such as maize and beans.


In [ ]:
# Read every seasonal file, then stack the years that belong to the same part of the questionnaire.
sas_parts = defaultdict(list)
data_dictionary = []
read_log = []
CSV_YEARS = {2023, 2024, 2025}


def read_delimited(path):
    # 2023 is comma-separated. 2024 and 2025 are tab-separated.
    # A few 2023 cells contain one byte that is not valid text. It is kept as a replacement mark so the row is not lost.
    text = path.read_bytes().decode("utf-8-sig", errors="replace")
    first = text.splitlines()[0] if text else ""
    separator = "\t" if "\t" in first else ","
    frame = pd.read_csv(io.StringIO(text), dtype=str, keep_default_na=False, sep=separator)
    frame.columns = [str(column).replace("\ufeff", "").strip() for column in frame.columns]
    return clean_text_columns(frame)


def drop_empty_columns(frame):
    # A column that is blank on every row does not help the person reading the table.
    filled = [column for column in frame.columns if frame[column].notna().any()]
    return frame[filled]


for year in (2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025):
    folder = RAW / "sas" / str(year)
    paths = sorted(folder.glob("*.csv" if year in CSV_YEARS else "*.dta"))
    print(f"\n{year}")
    for path in paths:
        if path.stat().st_size < 1000:
            print(f"  skipped empty file {path.name}")
            continue
        season, module = season_of(path.name), module_of(path.name)
        if year in CSV_YEARS:
            frame = read_delimited(path)
            labels = [""] * frame.shape[1]
        else:
            frame, labels = read_stata(path)
        prepared = prepare_seasonal_file(frame, year, season, module, path.name)
        sas_parts[module].append(prepared)
        data_dictionary.extend(dictionary_rows("SAS", year, season, module, path.name, list(frame.columns), labels))
        read_log.append({"year": year, "season": season, "module": module, "rows": len(prepared)})
        print(f"  season {season or '-'}: {module.replace('_', ' ')}, {len(prepared):,} rows")

sas_tables = {
    module: drop_empty_columns(pd.concat(frames, ignore_index=True, sort=False))
    for module, frames in sas_parts.items()
}

crop = sas_tables["crop_production"]
print("\nCrop records by year and season. These are interviews, not national totals.")
display(
    crop.groupby(["year", "season"], dropna=False)
    .agg(rows=("segment_id", "size"), segments=("segment_id", "nunique"), blank_weight=("plot_weight", lambda s: int(s.isna().sum())))
    .reset_index()
)

unmatched = crop.loc[crop["province_check"].eq("district_not_in_boundary"), ["year", "district_survey"]]
if unmatched.empty:
    print("Every district on the crop file matched an official district.")
else:
    print("District texts on the crop file that did not match. The rows are still kept.")
    display(unmatched.value_counts().rename_axis(["year", "district_survey"]).reset_index(name="rows").head(10))

preview = ["year", "season", "province", "district", "crop_name", "harvested_area_ha", "yield_kg_per_ha", "production_kg", "plot_weight"]
print("A few 2025 crop rows, so you can see the plain columns.")
display(crop.loc[crop["year"].eq(2025), preview].head(6))


## Household survey, 2024

The annual household survey is not used. CropWatch follows the seasonal survey, prices, rain, vegetation, and trade. This survey is left out so the folder stays small.

In [ ]:
# Left out on purpose. The save step below expects this empty dictionary.
ahs_tables = {}
print("Annual household survey: not used.")

## Food security survey, 2018

The 2018 food-security survey is one year only. It is not needed for the crop, rain, price, or trade pages, so it is left out.

In [ ]:
# Left out on purpose. The save step below expects this empty dictionary.
cfsva_tables = {}
print("Food security survey: not used.")

## Market prices

The two parquet files are one WFP price series, split into a training piece and a test piece. They are stacked, and an exact repeated row is kept once. The district comes from `admin2`. The price is in Rwandan francs, and `usdprice` is the same observation in US dollars. The unit column says whether the price is per kilogram or per sack, so those are not compared as if they were the same.


In [ ]:
price_frames = [pd.read_parquet(locate(name)) for name in ("train-00000-of-00001.parquet", "test-00000-of-00001.parquet")]
prices = pd.concat(price_frames, ignore_index=True)
before = len(prices)
prices = prices.drop_duplicates().reset_index(drop=True)
prices.insert(0, "source", "WFP HDX")
prices = attach_geography(prices, "admin1", "admin2")
for column in ("price", "usdprice", "latitude", "longitude"):
    if column in prices.columns:
        prices[column] = pd.to_numeric(prices[column], errors="coerce")
prices["date"] = pd.to_datetime(prices["date"], errors="coerce")
print(f"Price rows: {len(prices):,} after removing {before - len(prices):,} exact duplicates.")
print(f"Dates: {prices['date'].min().date()} to {prices['date'].max().date()}.")
print(f"Markets: {prices['market'].nunique():,}. Commodities: {prices['commodity'].nunique():,}.")
display(prices.groupby(prices["date"].dt.year).size().rename_axis("year").reset_index(name="rows"))


## Monthly rainfall

This file is already one row per district and month. It is CHIRPS daily rain added up to the month, in millimetres. The official district name is attached. September 2026 onward is not in the file.


In [ ]:
rain = pd.read_csv(locate("hingasmart_rwanda_chirps_monthly_district_rainfall_2018_2026.csv"))
rain = attach_geography(rain, "province", "district_name")
rain["year"] = pd.to_numeric(rain["year"], errors="coerce").astype("int64")
rain["month"] = pd.to_numeric(rain["month"], errors="coerce").astype("int64")
rain["rainfall_mm"] = pd.to_numeric(rain["rainfall_mm"], errors="coerce")
print(f"Rainfall rows: {len(rain):,}. Districts: {rain['district'].nunique()}.")
print(f"Months: {rain['year'].min()}-{rain['month'].min():02d} to {rain['year'].max()}-{rain['month'].max():02d}.")
unmatched_rain = rain.loc[rain["province_check"].eq("district_not_in_boundary"), "district_name"]
if unmatched_rain.empty:
    print("Every rainfall district matched an official district.")
else:
    print("Rainfall district names that did not match.")
    display(unmatched_rain.value_counts().rename_axis("district_name").reset_index(name="rows"))


## Exports by commodity and partner

Each download is one year of Rwanda's exports. A row is one commodity and one partner country. A partner called World is the total, so it is not added to the countries. The year 2023 stays from the file prepared earlier, because it was not in the new downloads. There is no 2026 export file yet.

In [ ]:
# Read every yearly export file. Pandas can shift these columns, so the standard reader is used.
csv.field_size_limit(50_000_000)
trade_columns = {
    "refYear": "year",
    "reporterISO": "reporter_iso",
    "reporterDesc": "reporter",
    "flowDesc": "flow",
    "partnerISO": "partner_iso",
    "partnerDesc": "partner",
    "cmdCode": "commodity_code",
    "cmdDesc": "commodity",
    "qtyUnitAbbr": "quantity_unit",
    "qty": "quantity",
    "netWgt": "net_weight_kg",
    "primaryValue": "trade_value_usd",
}
trade_files = sorted((ROOT / "data" / "source").glob("TradeData_*.csv"))
if not trade_files:
    raise FileNotFoundError("Put the yearly TradeData csv files in data/source.")
frames = []
for path in trade_files:
    with path.open(encoding="utf-8-sig", newline="") as handle:
        frame = pd.DataFrame(list(csv.DictReader(handle)))
    keep = [column for column in trade_columns if column in frame.columns]
    frame = frame[keep].rename(columns=trade_columns)
    frame = frame.loc[frame["reporter_iso"].eq("RWA")]
    frames.append(frame)
    print(path.name, "rows", f"{len(frame):,}", "years", sorted(frame["year"].unique()))
trade = pd.concat(frames, ignore_index=True)
years_now = set(trade["year"].astype(str))
earlier_file = OUT / "trade" / "exports_by_partner.csv"
if earlier_file.exists() and "2023" not in years_now:
    earlier = pd.read_csv(earlier_file, dtype=str)
    earlier = earlier.loc[earlier["year"].eq("2023")]
    trade = pd.concat([trade, earlier], ignore_index=True)
    print("Kept 2023 from the earlier file:", f"{len(earlier):,}")
for column in ("year", "quantity", "net_weight_kg", "trade_value_usd"):
    if column in trade.columns:
        trade[column] = pd.to_numeric(trade[column], errors="coerce")
print("Export rows:", f"{len(trade):,}", "Years:", sorted(trade["year"].dropna().astype(int).unique()))
display(trade.head())

## Published tables for 2026

Seasons A and B of 2026 arrived as the report tables, not as the interview files. Each published sheet is saved as its own table. Sheets that only hold a table of contents are skipped. Season C of 2026 is not in the folder.


In [ ]:
import openpyxl

sas2026 = {}
published = [
    ("A", "Tables_2026 Season A (1).xlsx"),
    ("B", "Tables_2026 Season B (1).xlsx"),
]
for season, filename in published:
    book = openpyxl.load_workbook(locate(filename), read_only=True, data_only=True)
    kept = 0
    for sheet in book.sheetnames:
        if not sheet.lower().startswith("table"):
            continue
        rows = [
            row for row in book[sheet].iter_rows(values_only=True)
            if any(cell not in (None, "") for cell in row)
        ]
        if not rows:
            continue
        width = max(len(row) for row in rows)
        frame = pd.DataFrame([list(row) + [None] * (width - len(row)) for row in rows])
        frame.insert(0, "sheet", sheet)
        frame.insert(0, "season", season)
        frame.insert(0, "year", 2026)
        key = f"season_{season.lower()}_{sheet.lower().replace(' ', '_')}"
        sas2026[key] = frame
        kept += 1
    book.close()
    print(f"2026 season {season}: {kept} tables")


## Save the tables

CSV is there so you can open a file in Excel. Parquet is the same table, smaller, and is the better one to read next time in Python. The data dictionary lists every question code and its wording. The coverage table is the 2018-to-now timeline, including the years that have no file.


In [ ]:
# Write every prepared table, then a short catalogue and the year-by-year coverage.
def save_table(folder, name, frame):
    folder.mkdir(parents=True, exist_ok=True)
    csv_path = folder / f"{name}.csv"
    frame.to_csv(csv_path, index=False)
    try:
        frame.to_parquet(folder / f"{name}.parquet", index=False)
        parquet = "yes"
    except Exception as exc:
        parquet = exc.__class__.__name__
    return {
        "folder": str(folder.relative_to(ROOT)).replace("\\", "/"),
        "table": name,
        "rows": len(frame),
        "columns": frame.shape[1],
        "csv_mb": round(csv_path.stat().st_size / 1e6, 1),
        "parquet": parquet,
    }

catalog_rows = []
for module, frame in sas_tables.items():
    catalog_rows.append(save_table(OUT / "sas", module, frame))
for slug, frame in ahs_tables.items():
    catalog_rows.append(save_table(OUT / "ahs2024", slug, frame))
for role, frame in cfsva_tables.items():
    catalog_rows.append(save_table(OUT / "cfsva2018", role, frame))
catalog_rows.append(save_table(OUT / "prices", "market_prices", prices))
catalog_rows.append(save_table(OUT / "rainfall", "district_month", rain))
catalog_rows.append(save_table(OUT / "trade", "exports_by_partner", trade))
for name, frame in sas2026.items():
    catalog_rows.append(save_table(OUT / "sas2026", name, frame))

districts.drop(columns=["district_key"]).sort_values(["province_code", "district"]).to_csv(OUT / "district_lookup.csv", index=False)
dictionary = pd.DataFrame(data_dictionary).drop_duplicates(["survey", "year", "file", "column"])
dictionary.to_csv(OUT / "data_dictionary.csv", index=False)

# The timeline of the project. Empty years are written down so they are not mistaken for zeros.
coverage = pd.DataFrame([
    {"year": year, "seasonal_survey": "microdata", "notes": ""}
    for year in range(2018, 2026)
] + [{
    "year": 2026,
    "seasonal_survey": "published tables, seasons A and B",
    "notes": "Season C and the 2026 interview files are not in the folder",
}])
coverage["food_security"] = "not used"
coverage["household_survey"] = "not used"
coverage["prices"] = "WFP market prices"
coverage["rainfall"] = "monthly district rainfall"
coverage.loc[coverage["year"].eq(2026), "rainfall"] = "monthly district rainfall through August"
coverage["trade"] = "exports by commodity and partner"
coverage.loc[coverage["year"].eq(2026), "trade"] = ""
coverage.to_csv(OUT / "coverage_2018_to_now.csv", index=False)

catalog = pd.DataFrame(catalog_rows)
catalog.to_csv(OUT / "catalog.csv", index=False)
print("Saved to", OUT)
display(coverage)
display(catalog)
print(f"Question dictionary: {len(dictionary):,} rows.")

## Match the names

The survey and the price list do not always use the same crop name. This step keeps a pair only when it is the same product. Flour stays apart from the harvest. Kinyarwanda names are the everyday farming names, written next to the English name.

In [ ]:
# These steps can run after the big survey clean, or on their own when the cleaned files are already saved.
import sys
from pathlib import Path

HERE = Path.cwd()
PROJECT = HERE if (HERE / "src").exists() else HERE.parent
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from src.prepare_matches import main as match_crop_names
from src.prepare_step2 import clean_calendar, rain_anomaly, slim_vegetation, stack_trade

match_crop_names()
slim_vegetation()
clean_calendar()
stack_trade()
rain_anomaly()

## Remove the copies that are no longer needed

The original zip files stay in `data/source`, so the survey can be prepared again. The unpacked folder is only a second copy of those zips, so it is removed after the cleaned tables exist. The old 8-row national export total is removed too, because the partner file replaced it. The heavy map shapes inside the vegetation download are removed after the district numbers are saved.

In [ ]:
import shutil
import pandas as pd

removed = []
raw_copy = PROJECT / "data" / "raw"
if raw_copy.exists():
    shutil.rmtree(raw_copy)
    removed.append("data/raw")

leftovers = [
    PROJECT / "data" / "processed" / "trade" / "national_exports.csv",
    PROJECT / "data" / "processed" / "trade" / "national_exports.parquet",
    PROJECT / "data" / "source" / "rwanda_ndre_2018_2025.csv",
]
vegetation_saved = PROJECT / "data" / "processed" / "vegetation" / "district_year.csv"
if vegetation_saved.exists():
    leftovers.append(PROJECT / "data" / "source" / "rwanda_ndre_30_districts_2018_2026.csv")
for leftover in leftovers:
    if leftover.exists():
        leftover.unlink()
        removed.append(str(leftover.relative_to(PROJECT)).replace("\\", "/"))

cache = PROJECT / "src" / "__pycache__"
if cache.exists():
    shutil.rmtree(cache)
    removed.append("src/__pycache__")

catalog_path = PROJECT / "data" / "processed" / "catalog.csv"
if catalog_path.exists():
    catalog_now = pd.read_csv(catalog_path)
    if "table" in catalog_now.columns:
        catalog_now = catalog_now.loc[catalog_now["table"].ne("national_exports")]
        catalog_now.to_csv(catalog_path, index=False)

if removed:
    print("Removed:")
    for name in removed:
        print("-", name)
else:
    print("Nothing left to remove. The cleaned tables and the original zip files are still here.")

## What to open next

- `data/processed/sas/crop_production.csv` is the survey, 2018 to 2025. One row is one crop on a plot. A national quantity is `production_kg` times `plot_weight`, done separately for each season.
- `data/processed/prices/market_prices.csv` is the market prices. Compare prices only when the unit is the same.
- `data/processed/lookups/crop_price_match.csv` is the list of crops that match a price, with the Kinyarwanda name.
- `data/processed/lookups/crop_calendar_zones.csv` is planting and harvest by farming zone. A zone is not a district.
- `data/processed/rainfall/district_month_anomaly.csv` compares each month with the usual rain for that month. Usual is the average of 2018 to 2025.
- `data/processed/vegetation/district_year.csv` is one vegetation number for each district and year.
- `data/processed/trade/exports_by_partner.csv` is exports by commodity and partner. Do not add the World rows to the country rows.

The app reads these tables. Open it and choose English or Ikinyarwanda.